In [ ]:
# 导入依赖与工具
import torch
import matplotlib.pyplot as plt
from model import ResNet18_CIFAR
from data_utils import testloader, device, classes, epsilon, pgd_alpha, pgd_steps
from eval import fgsm_attack, pgd_attack, deepfool_attack
import os

# 加载干净模型
model = ResNet18_CIFAR().to(device)
if os.path.exists("models/resnet18_clean.pth"):
    model.load_state_dict(torch.load("models/resnet18_clean.pth", map_location=device))
    print("模型加载完成")

In [ ]:
# 单张样本生成FGSM对抗样本并展示
imgs, labels = next(iter(testloader))
img, label = imgs[0:1].to(device), labels[0].to(device)
adv_fgsm = fgsm_attack(model, img, label, epsilon)

with torch.no_grad():
    pred_clean = model(img).argmax(1).item()
    pred_adv = model(adv_fgsm).argmax(1).item()

plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
plt.imshow(img[0].cpu().permute(1,2,0).numpy())
plt.title(f"原图：{classes[label.item()]}")
plt.subplot(1,2,2)
plt.imshow(adv_fgsm[0].cpu().permute(1,2,0).numpy())
plt.title(f"FGSM对抗样本：预测={classes[pred_adv]}")
plt.show()
print(f"原图真实标签：{classes[label.item()]}, 模型预测：{classes[pred_clean]}")
print(f"对抗样本模型预测：{classes[pred_adv]}")

In [ ]:
# 批量测试攻击成功率
from eval import test_attack
asr, adv_acc, _ = test_attack(model, testloader, pgd_attack, device, epsilon=epsilon, alpha=pgd_alpha, steps=pgd_steps)
print(f"PGD攻击成功率ASR={asr:.2f}%, 对抗样本准确率={adv_acc:.2f}%")